# Forty-one million parameters, and the tricks that make them train

Decoder-only blocks, tied embeddings, warmup, and a logits-based loss — the most compute-intensive run in the whole course.

**Runs on:** GPU required — ~6 hours on a Colab T4, ~1 hour on an A100 &nbsp;·&nbsp; **Slides:** [Chapter 16 — Text Generation](../../../course-web-slides/ch16/index.html) &nbsp;·&nbsp; **Section:** 01 — Training a mini-GPT

---

## The decoder block, minus cross-attention

In [ ]:
import keras
from keras import layers, ops

class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(
            num_heads, key_dim, dropout=0.1)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()
        self.dropout = layers.Dropout(0.1)

    def call(self, inputs):
        residual = x = inputs
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = self.dropout(x)
        x = self.self_attention_layernorm(x + residual)

        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = self.dropout(x)
        x = self.feed_forward_layernorm(x + residual)
        return x

Chapter 15's decoder with cross-attention removed and dropout added **inside** each block — chapter 15 stacked one layer and got away with one dropout at the end; here we stack eight.

`use_causal_mask=True` is the only thing standing between this model and reading its own labels.

## Tied embeddings

In [ ]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs, reverse=False):
        if reverse:
            token_embeddings = self.token_embeddings.embeddings
            return ops.matmul(inputs, ops.transpose(token_embeddings))
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        return (self.token_embeddings(inputs)
                + self.position_embeddings(positions))

vocab_size, hidden_dim = 32000, 512
saved = vocab_size * hidden_dim
print(f"one matrix used twice saves {saved:,} parameters")
print(f"= {saved / 41_000_000:.0%} of a 41M-parameter model")

The two largest weights in a Transformer are both vocabulary-shaped: the token embedding **(vocab, hidden)** and the output projection **(hidden, vocab)**. They are transposes in shape; making them transposes in **value** works well.

Think of the output as a *reverse embedding*: hidden space back to token space.

## The model

In [ ]:
keras.config.set_dtype_policy("mixed_float16")

intermediate_dim, num_heads, num_layers = 2056, 8, 8
sequence_length = 256

inputs = keras.Input(shape=(None,), dtype="int32", name="inputs")
embedding = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)
x = embedding(inputs)
x = layers.LayerNormalization()(x)
for _ in range(num_layers):
    x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(x)
outputs = embedding(x, reverse=True)
mini_gpt = keras.Model(inputs, outputs)

print(f"{mini_gpt.count_params():,} parameters")
print("GPT-1 had 117 million; GPT-3 had 175 billion.")

> **Note** — `mixed_float16` trades some numerical fidelity for roughly 2× speed. Chapter 18 explains what it is doing and why loss scaling goes with it.

## Warmup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class WarmupSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self):
        self.rate = 2e-4
        self.warmup_steps = 1_000.0

    def __call__(self, step):
        step = ops.cast(step, dtype="float32")
        scale = ops.minimum(step / self.warmup_steps, 1.0)
        return self.rate * scale

schedule = WarmupSchedule()
xs = range(0, 5000, 50)
ys = [float(ops.convert_to_numpy(schedule(s))) for s in xs]
plt.figure(figsize=(6.5, 3.6))
plt.plot(xs, ys, lw=1.8)
plt.xlabel("train step"); plt.ylabel("learning rate")
plt.title("Linear warmup over 1,000 steps, then flat")
plt.show()

Stack many Transformer layers and **exploding gradients** are easy to hit — parameters update too fast and the loss never converges. A linear ramp keeps the earliest updates small.

**Plot the schedule before training.** A wrong schedule is invisible in the loss curve until it is far too late.

## What is a logit?

In [ ]:
print("The output projection has NO softmax activation.")
print()
print("Its outputs are unnormalized log probabilities. Exponentiate and")
print("normalize -- which is all softmax does -- and you get probabilities.")
print()
print("  softmax IN THE MODEL:  Dense(n, activation='softmax')")
print("                       + SparseCategoricalCrossentropy()")
print()
print("  softmax IN THE LOSS:   Dense(n)")
print("                       + SparseCategoricalCrossentropy(from_logits=True)")
print()
print("The second is more numerically stable and far easier to sample from,")
print("which is what the next notebook needs.")

## Training

In [ ]:
num_epochs = 8
num_train_batches, num_val_batches = 28873, 500
steps_per_epoch = num_train_batches // num_epochs

mini_gpt.compile(
    optimizer=keras.optimizers.Adam(schedule),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
mini_gpt.fit(
    train_ds,
    validation_data=val_ds,
    epochs=num_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_steps=num_val_batches,
    callbacks=[keras.callbacks.ModelCheckpoint("mini_gpt.keras",
                                               save_best_only=True)],
    verbose=2,
)

> ⚠️ **This is the most computationally expensive run in the book.** Six hours on a free Colab T4. Set it going and take a break — or cut `steps_per_epoch` for a quick experiment and accept a worse model.

## The honest reading of the result

About **36% next-token accuracy** — and the validation loss is **still falling** after the last epoch.

This model is not converged and we know it. That is unsurprising with a hundred times fewer training steps than GPT-1, and it is worth stating plainly: most published curves stop where the budget ran out, not where the model stopped improving.

**The gap to a modern LLM is 100× the parameters and 1,000× the steps — and nothing else.** The recipe above is the one everyone is using.

---

## What to take away

- Decoder-only: chapter 15's block with cross-attention removed and dropout added inside.
- Tied embeddings use one matrix forward and transposed, saving a large share of the parameters.
- Warmup prevents exploding gradients; plot the schedule before training.
- `from_logits=True` is more stable and much easier to sample from.